# Q1. 199. Bag of words vectorization





Turn raw text into numbers using Bag-of-Words (BoW) vectorization, a simple NLP preprocessing step that counts how often each word appears. Given a fixed vocabulary, represent each sentence as a count vector where each entry corresponds to a vocabulary word.

Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
⌄
⌄
def bag_of_words_vectorize(texts, vocabulary):
    """
    Builds Bag-of-Words vectors for a list of texts using a fixed vocabulary.

    Args:
        texts (np.ndarray): Input sentences/documents.
        vocabulary (np.ndarray): Unique tokens defining the vector order.

    Returns:
        np.ndarray: A matrix of shape (len(texts), len(vocabulary)),
        where result[i][j] is the count of vocabulary[j] in texts[i].
    """
    # Your solution below
Rules:

Tokenize each text by lowercasing and splitting on whitespace.
For each text, count occurrences of each vocabulary token and output counts in the exact vocabulary order.
Words not in vocabulary are ignored.
Return the result as a NumPy array.
Do not use any prebuilt vectorizers (e.g., scikit-learn).
Use only NumPy and built-in Python libraries.
Example
python
Copy
1
2
3
4
texts = ["I like cats", "I like like dogs"]
vocabulary = ["i", "like", "cats", "dogs"]

bag_of_words_vectorize(texts, vocabulary)
Output:

python
Copy
1
2
3
4
⌄
array([
    [1, 1, 1, 0],
    [1, 2, 0, 1]
])


In [ ]:
import numpy as np

def bag_of_words_vectorize(texts, vocabulary):
    """
    Builds Bag-of-Words vectors for a list of texts using a fixed vocabulary.

    Args:
        texts (np.ndarray): Input sentences/documents.
        vocabulary (np.ndarray): Unique tokens defining the vector order.

    Returns:
        np.ndarray: A matrix of shape (len(texts), len(vocabulary)),
        where result[i][j] is the count of vocabulary[j] in texts[i].
    """
    # Map each vocabulary token to its index for quick lookup
    # Ensure we use standard strings for consistent dictionary lookup
    vocab_index = {str(token): idx for idx, token in enumerate(vocabulary)}
    vocab_size = len(vocabulary)

    # Initialize the result matrix using NumPy
    result = np.zeros((len(texts), vocab_size), dtype=int)

    # Process each text
    for i, text in enumerate(texts):
        # Tokenize by lowercasing and splitting on whitespace
        # Cast to string to handle numpy string types safely
        tokens = str(text).lower().split()
        # Count occurrences of vocabulary tokens
        for token in tokens:
            if token in vocab_index:
                j = vocab_index[token]
                result[i, j] += 1

    return result

result = bag_of_words_vectorize(texts, vocabulary)
result

# Q2. 624. Attention masking




Implement causal attention masking for a self-attention score matrix in NLP, so each token can only attend to itself and earlier tokens. You’ll take raw attention scores and apply a mask that forces “future” positions to have zero probability after softmax.

The masked attention is:

A
=
softmax
(
S
+
M
)
A=softmax(S+M)
where 
S
S is the score matrix and 
M
i
j
=
0
M 
ij
​
 =0 if 
j
≤
i
j≤i, otherwise 
M
i
j
=
−
10
9
M 
ij
​
 =−10 
9
 .

Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
⌄
⌄
def masked_softmax(scores):
    """
    Applies a causal (lower-triangular) mask to attention scores and then computes softmax.

    Args:
        scores (np.ndarray): Attention score matrix S of shape (n, n),
            where scores[i][j] is the raw score for token i attending to token j.

    Returns:
        np.ndarray: Attention probability matrix A of shape (n, n),
            where A[i][j] is the masked softmax probability.
    """
    # Your solution below
Rules:

Apply a causal mask so positions where j > i are masked out (future tokens).
Use a numerically stable softmax per row (subtract the row max before exp).
Use -1e9 (or another very large negative number) for masked positions before softmax.
Return the result as a NumPy array.
Use only NumPy (and built-in Python libraries if needed).
Example
python
Copy
1
2
3
4
5
6
7
⌄
scores = np.array([
    [1.0, 2.0, 3.0],
    [1.0, 2.0, 3.0],
    [1.0, 2.0, 3.0],
])

masked_softmax(scores)
Output:

python
Copy
1
2
3
4
5
⌄
[
    [1.0, 0.0, 0.0],
    [0.2689, 0.7311, 0.0],
    [0.0900, 0.2447, 0.6652]
]
Input Signature

In [ ]:
import numpy as np


def masked_softmax(scores):
    """
    Applies a causal (lower-triangular) mask to attention scores and then
    computes softmax.

    Args:
        scores (np.ndarray): Attention score matrix S of shape (n, n),
            where scores[i][j] is the raw score for token i attending to
            token j.

    Returns:
        np.ndarray: Attention probability matrix A of shape (n, n),
            where A[i][j] is the masked softmax probability.
    """
    n = scores.shape[0]

    # Create a causal mask with large negative values for future positions
    mask = np.triu(np.full((n, n), -1e9), k=1)
    masked_scores = scores + mask

    # Numerically stable softmax per row
    row_max = np.max(masked_scores, axis=1, keepdims=True)
    stabilized = masked_scores - row_max
    exp_scores = np.exp(stabilized)
    row_sum = np.sum(exp_scores, axis=1, keepdims=True)

    return exp_scores / row_sum

result = masked_softmax(scores)
result

# Q3. 318. Positional encoding




Implement sinusoidal positional encoding so you can add position information to token embeddings in an NLP model. You’ll generate a matrix where each position has a deterministic encoding based on sine and cosine waves.

The encoding is defined as:

PE
(
p
o
s
,
2
i
)
=
sin
⁡
(
p
o
s
10000
2
i
d
)
,
PE
(
p
o
s
,
2
i
+
1
)
=
cos
⁡
(
p
o
s
10000
2
i
d
)
PE(pos,2i)=sin( 
10000 
d
2i
​
 
 
pos
​
 ),PE(pos,2i+1)=cos( 
10000 
d
2i
​
 
 
pos
​
 )
where 
p
o
s
pos is the token index (starting at 0), 
i
i is the dimension index, and 
d
d is the model dimension.

Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
⌄
⌄
import numpy as np

def positional_encoding(seq_len, d_model):
    """
    Builds a sinusoidal positional encoding table.

    Args:
        seq_len (int): Number of token positions in the sequence.
        d_model (int): Embedding dimension (number of features per token).

    Returns:
        np.ndarray: Positional encoding matrix of shape (seq_len, d_model),
        where row `pos` is the encoding for that token position.
    """
    # Your solution below
Rules:

Use the sinusoidal formula above with positions pos = 0..seq_len-1.
Even dimensions use sin, odd dimensions use cos.
Return the result as a NumPy array.
Use only NumPy and Python built-in libraries (no PyTorch/TensorFlow).
Aim for a vectorized NumPy implementation (avoid deeply nested loops if you can).
Example
python
Copy
1
2
3
4
seq_len = 3
d_model = 4

positional_encoding(seq_len, d_model)
Output:

python
Copy
1
2
3
4
5
⌄
array([
  [0.0, 1.0, 0.0, 1.0],
  [0.84147, 0.54030, 0.0099998, 0.99995],
  [0.90930, -0.41615, 0.0199987, 0.99980]
])


In [ ]:
import numpy as np


def positional_encoding(seq_len, d_model):
    """
    Builds a sinusoidal positional encoding table.

    Args:
        seq_len (int): Number of token positions in the sequence.
        d_model (int): Embedding dimension (number of features per token).

    Returns:
        np.ndarray: Positional encoding matrix of shape
        (seq_len, d_model), where row `pos` is the encoding for that
        token position.
    """
    # Create position indices (seq_len, 1)
    positions = np.arange(seq_len)[:, np.newaxis]
    # Compute scaling factors for even indices
    div_term = np.exp(
        np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model)
    )
    # Initialize positional encoding matrix
    pe = np.zeros((seq_len, d_model))
    # Apply sine to even indices
    pe[:, 0::2] = np.sin(positions * div_term)
    # Apply cosine to odd indices
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe

result = positional_encoding(seq_len, d_model)
result